### Backpropagation

Just dynamical programming...

In [123]:
import sys, time, numpy as np, plotly.express as px
from sklearn.model_selection import train_test_split
SEED = 6767676

## Så, dette er basicly backpropagation.

la oss si at vi har input ved x, hidden layers y, og output z.

$\begin{matrix}
  & y_{11} & y_{21} & \\
x & y_{12} & y_{22} & z \\
  & y_{13} & y_{23} &
\end{matrix}$

### For en node $y_{1i}$ ønsker vi å finne $\frac{\partial z}{\partial y_{1i}}$, hvor z er final prediction
### $\frac{\partial z}{\partial y_{1i}}$ forteller oss hva en endring ved noden $y_{1i}$ gjør med sluttresultatet $z$.
Dette kan vi bruke for til å justere noder for å nå en endring som er lik $y - prediksjon$, altså om $y=5$ & $prediksjon=2$ vil vi endre modellen vår så vi får eksakt +3 i endring. og $prediksjon=2+3=5$
### La oss så si for et nivå i det nevrale nettverket:
- $y_{2j} = \sum_{i} w_{1,i,j} \cdot y_{1i} \ + b_j$
- da er $\frac{\partial z}{\partial y_{1i}} = \sum_{j} \frac{\partial y_{2j}}{\partial y_{1i}} \frac{\partial z}{\partial y_{2j}} = \sum_{j} \frac{\partial \sum_{k} w_{1,k,j} \cdot y_{1k} \ + b_j}{\partial y_{1i}} \frac{\partial z}{\partial y_{2j}} = \sum_{j} w_{1ij} \frac{\partial z}{\partial y_{2j}}$

Så vi lagrer:
- la $layer = l$, $i$ er indexen i layer, f.eks $y_{11}$ er layer nummer 1 node nummer 1, $y_{22}$ er layer 2 node 2
- $dp[l][i] = \frac{\partial z}{\partial y_{l,i}} = \sum_{j} w_{l,i,j} \cdot \frac{\partial z}{\partial y_{l+1,j}}$


### En viktig ting å nevne
Vi bruker egentlig ReLU -> $max(0, wx + b)$
- $y_{l+1,j} = ReLU\left(\sum_{i} w_{l,i,j} \cdot y_{l,i} \ + b_j\right) = max\left(0, \sum_{i} w_{l,i,j} \cdot y_{l,i} \ + b_j\right)$

Så vi må egentlig ta:

- $\frac{\partial z}{\partial y_{l,i}} = \sum_{j} \frac{\partial y_{l+1,j}}{\partial y_{l,i}} \cdot \frac{\partial z}{\partial y_{l+1,j}} = \sum_{j} \frac{\partial ReLU\left(\sum_{k} w_{l,k,j} \cdot y_{l,k} \ + b_j\right)}{\partial y_{l,i}} \cdot \frac{\partial z}{\partial y_{l+1,j}} = \sum_{j} w_{l,i,j} \cdot \frac{\partial z}{\partial y_{l+1,j}}$ men kun for $j$ hvor $\sum_{k} w_{l,k,j} \cdot y_{l,k} \ + b_j > 0$ ellers $0$

Og dette er basicly en medium leetcode oppgave når man vet hva man skal gjøre!

In [124]:
def derivatives(CNN, z):
    dp = [[0.0] * len(CNN[i]) for i in range(CNN.layers)]
    dp[-1] = [z]
    for l in range(CNN.layers-2, -1, -1):
        n = len(CNN[l])
        m = len(CNN[l + 1])

        for i in range(n):
            node = CNN[l][i]
            for j in range(m):
                wlij = node.w[j]
                next_node = CNN[l + 1][j] 
                
                dReLu = 1 if next_node.pre > 0 or l == CNN.layers - 2 else 0
                
                dp[l][i] += dReLu * wlij * dp[l + 1][j]
    return dp

### Oppdatering av konstanter og variabler fra deriverte
Vi lar basicly $z = y - prediksjon = error$

Vi ønsker å gjøre $w_{l,i,j}$ -= $\delta \cdot \frac{\partial error}{\partial w_{l,i,j}}$ Hvor $\delta$ er et veldig lite tall.
På denne måten oppdaterer vi $w_{l,i,j}$ til å minimiere error ~ error blir mindre.


$\frac{\partial z}{\partial b_{l,i}} = \sum_{j} 1 \cdot \frac{\partial z}{\partial y_{l+1,j}} = \sum_{j} dp[l+1][j]$ Denne er ganske enkel, så vi gjør $b_{l,i}$ -= $\delta \cdot dp[l+1][l]$

In [125]:
def update(dp, CNN, delta):
    for l in range(CNN.layers-1):
        n = len(CNN[l])
        m = len(CNN[l+1])
        for j in range(m):
            next_node = CNN[l + 1][j]
            dReLu = 1 if next_node.pre > 0 or l == CNN.layers - 2 else 0
            gradient = dReLu * dp[l + 1][j]
            next_node.b -= delta * gradient
            for i in range(n):
                node = CNN[l][i]
                node.w[j] -= delta * gradient * node.z

## For en verdi $x$ gjør vi prediksjon, aka forward step fra input to output

In [126]:
ReLU = lambda x: max(0, x)

def forward(CNN, x):
    CNN[0][0].z = x
    for l in range(1, CNN.layers):
        cur, prev = CNN[l], CNN[l-1]
        n, m = len(cur), len(prev)
        for i in range(n):
            cur[i].z = cur[i].b
            for j in range(m):
                w, z = prev[j].w[i], prev[j].z
                cur[i].z += w * z
            cur[i].pre = cur[i].z
            if l < CNN.layers - 1: cur[i].z = ReLU(cur[i].z)
    return sum(O.z for O in CNN[-1])

### Strukturen på nettverket.
Vi trekker tilfeldige tall fra en normalfordeling med $\mu = 0$ og $\sigma = 1$, så vi får både initielle positive og negative weights, (små).

In [ ]:
rng = np.random.default_rng(SEED)
class Neuron:
    def __init__(self, width):
        self.w = [rng.normal(0, 1) for _ in range(width)] # vekter fra node i til node j. som oppdateres fra gradient descent.
        self.b, self.z, self.pre = 0, 0, 0
class CNN:
    def __init__(self, depth, width):
        self.matrix = (
            [[Neuron(width)]] # x -> input
            + [[Neuron(width) for _ in range(width)] for _ in range(depth)] # y
            + [[Neuron(0)]] # z -> output
        )
        self.layers = len(self.matrix)
    def __len__(self):
        return len(self.matrix)
    def __getitem__(self, i):
        return self.matrix[i]

In [ ]:
rng = np.random.default_rng(SEED)
cnn = CNN(depth=3, width = 8)

n = 4000
delta = 0.00001

poly = lambda x: x**3 + x**2 + x + 1
X = rng.uniform(-4, 4, size=300)

X_tren, X_rem = train_test_split(X, train_size=0.3, random_state=SEED)
X_val, X_test = train_test_split(X_rem, train_size=0.5, random_state=SEED)
Y_tren, Y_val, Y_test = [poly(x) for x in X_tren], [poly(x) for x in X_val], [poly(x) for x in X_test]
for i in range(n):
    MSE = 0
    P = []
    for x, y in zip(X_tren, Y_tren):
        prediction = forward(cnn, x)
        dp = derivatives(cnn, prediction - y) # vil ha en endring eksakt lik forskjellen y - prediction
        update(dp, cnn, delta) # vi har deriverte i dp, bare bruk
        MSE += (prediction - y) ** 2 / len(X_tren)
        P.append(prediction)
    
    if i % 500 == 0:
        fig = px.scatter(x=X_tren, y=[Y_tren, P], labels={"value":"Y"}, title=f"MSE: {MSE}", height=200, width=800)
        fig.data[0].name = "x^3 + x^2 + x + 1"
        fig.data[1].name = "Prediction"
        fig.update_layout(margin=dict(l=0, r=0, t=30, b=0)).show()
        print(MSE)

pred_val = [forward(cnn, x) for x in X_val]
MSE_val = sum((p - y)**2 for y, p in zip(pred_val, Y_val)) / len(Y_val)

fig = px.scatter(x=X_val, y=[Y_val, pred_val], labels={"value":"Y"}, title=f"MSE Validation: {MSE_val}", height=200, width=800)
fig.data[0].name = "x^3 + x^2 + x + 1"
fig.data[1].name = "Prediction"
fig.update_layout(margin=dict(l=0, r=0, t=30, b=0)).show()





828.2949219385183


4.843888793746374


1.9525696168150772


1.182066568167834


0.8074225442639509


0.6797425145658273


0.5610080084061494


0.5109575623346646
